In [0]:
import re
import pandas as pd
RAW_DIR = "/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/data"
CLEAN_DIR = "/Workspace/Users/yashikumawat53@gmail.com/Drafts/ecommerce_project/cleaned_data"
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

In [0]:
def clean_orders(orders: pd.DataFrame) -> tuple[pd.DataFrame, dict]:

    df = orders.copy()
    issues = {}

    df["customer_id"] = df["customer_id"].replace(r"^\s*$", pd.NA, regex=True)
    missing_customer_mask = df["customer_id"].isna()
    issues["orders_missing_customer_id"] = int(missing_customer_mask.sum())

    df["customer_id_missing"] = missing_customer_mask
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")

    def parse_date(value):
        if pd.isna(value):
            return pd.NaT
        value = str(value).strip()
        parsed = pd.to_datetime(value, format="%Y-%m-%d %H:%M:%S", errors="coerce")
        if pd.isna(parsed):
            parsed = pd.to_datetime(value, format="%d-%m-%Y", errors="coerce")
        return parsed

    wrong_format_mask = ~df["order_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
    issues["orders_wrong_date_format"] = int(wrong_format_mask.sum())

    df["order_date"] = df["order_date"].apply(parse_date)
    unparseable_mask = df["order_date"].isna()
    issues["orders_unparseable_dates"] = int(unparseable_mask.sum())

    issues["orders_total_rows"] = len(df)
    return df, issues

In [0]:
def clean_products(products: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = products.copy()
    issues = {}

    original = df["product_name"]
    messy_mask = (
        original.str.strip().ne(original) 
        | original.str.contains(r"\s{2,}")  
        | original.ne(original.str.title())  
    )
    issues["products_messy_names"] = int(messy_mask.sum())

    df["product_name"] = (
        df["product_name"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    issues["products_total_rows"] = len(df)
    return df, issues

In [0]:
def validate_emails(customers: pd.DataFrame) -> list:
    invalid_mask = ~customers["email"].astype(str).str.match(EMAIL_RE)
    invalid_ids = customers.loc[invalid_mask, "customer_id"].tolist()
    return invalid_ids

In [0]:
def check_referential_integrity(order_items: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    valid_order_ids = set(orders["order_id"])
    orphan_mask = ~order_items["order_id"].isin(valid_order_ids)
    return order_items.loc[orphan_mask]

In [0]:
def main():
    import os
    os.makedirs(CLEAN_DIR, exist_ok=True)
    customers = pd.read_csv(f"{RAW_DIR}/customers.csv")
    products = pd.read_csv(f"{RAW_DIR}/products.csv")
    orders = pd.read_csv(f"{RAW_DIR}/orders.csv", dtype={"customer_id": "object"})
    order_items = pd.read_csv(f"{RAW_DIR}/order_items.csv")

    all_issues = {}

    clean_orders_df, order_issues = clean_orders(orders)
    all_issues.update(order_issues)

    clean_products_df, product_issues = clean_products(products)
    all_issues.update(product_issues)

    invalid_email_ids = validate_emails(customers)
    all_issues["customers_invalid_emails"] = len(invalid_email_ids)
    all_issues["customers_total_rows"] = len(customers)

    orphan_items = check_referential_integrity(order_items, orders)
    all_issues["order_items_orphan_rows"] = len(orphan_items)
    all_issues["order_items_negative_quantity"] = int((order_items["quantity"] < 0).sum())
    all_issues["order_items_discount_out_of_range"] = int(
        ((order_items["discount_percent"] < 0) | (order_items["discount_percent"] > 100)).sum()
    )
    all_issues["order_items_total_rows"] = len(order_items)

    clean_order_items_df = order_items.loc[~order_items.index.isin(orphan_items.index)].copy()
    customers.to_csv(f"{CLEAN_DIR}/customers.csv", index=False)
    clean_products_df.to_csv(f"{CLEAN_DIR}/products.csv", index=False)
    clean_orders_df.to_csv(f"{CLEAN_DIR}/orders.csv", index=False)
    clean_order_items_df.to_csv(f"{CLEAN_DIR}/order_items.csv", index=False)

    report_lines = [
        "DATA QUALITY REPORT",
        "=" * 60,
        "",
        "ORDERS",
        "-" * 60,
        f"  Total rows                         : {all_issues['orders_total_rows']}",
        f"  Missing customer_id                : {all_issues['orders_missing_customer_id']}",
        f"  Rows in wrong date format (DD-MM-YYYY): {all_issues['orders_wrong_date_format']}",
        f"  Dates that failed to parse entirely: {all_issues['orders_unparseable_dates']}",
        "",
        "PRODUCTS",
        "-" * 60,
        f"  Total rows                         : {all_issues['products_total_rows']}",
        f"  Messy product names (fixed)        : {all_issues['products_messy_names']}",
        "",
        "CUSTOMERS",
        "-" * 60,
        f"  Total rows                         : {all_issues['customers_total_rows']}",
        f"  Invalid emails                     : {all_issues['customers_invalid_emails']}",
        f"  Invalid email customer_ids         : {invalid_email_ids}",
        "",
        "ORDER_ITEMS",
        "-" * 60,
        f"  Total rows                         : {all_issues['order_items_total_rows']}",
        f"  Orphan rows (order_id not in orders): {all_issues['order_items_orphan_rows']} (removed from cleaned data)",
        f"  Negative quantity rows (returns)   : {all_issues['order_items_negative_quantity']}",
        f"  discount_percent out of [0,100]    : {all_issues['order_items_discount_out_of_range']}",
        "",
    ]

    report_text = "\n".join(report_lines)
    with open(f"{CLEAN_DIR}/data_quality_report.txt", "w") as f:
        f.write(report_text)

    print(report_text)


if __name__ == "__main__":
    main()

DATA QUALITY REPORT

ORDERS
------------------------------------------------------------
  Total rows                         : 900
  Missing customer_id                : 48
  Rows in wrong date format (DD-MM-YYYY): 58
  Dates that failed to parse entirely: 0

PRODUCTS
------------------------------------------------------------
  Total rows                         : 520
  Messy product names (fixed)        : 69

CUSTOMERS
------------------------------------------------------------
  Total rows                         : 550
  Invalid emails                     : 7
  Invalid email customer_ids         : [8, 220, 256, 284, 300, 473, 487]

ORDER_ITEMS
------------------------------------------------------------
  Total rows                         : 2240
  Orphan rows (order_id not in orders): 22 (removed from cleaned data)
  Negative quantity rows (returns)   : 85
  discount_percent out of [0,100]    : 0

